# AgeLens — 03 Validation

This notebook validates the diagnostic Phenotypic Age implementation produced by `02_data_preprocessing.ipynb`.

## Placement

```text
nhanes/
└── notebooks/
    ├── 00_setup_agelens.ipynb
    ├── 01_data_ingestion.ipynb
    ├── 02_data_preprocessing.ipynb
    └── 03_validation.ipynb
```

## Required input

```text
nhanes/data/interim/nhanes_2015_2018_preprocessed_diagnostic.parquet
```

## Validation domains

1. Sample-flow integrity
2. Age correlation by cycle and sensitivity sample
3. Survey-weighted descriptive summaries
4. Erratum-versus-Supplement comparison
5. Pre/post bridging effect on an identical sample
6. Top-code sensitivity
7. Age-20-plus sensitivity
8. Optional cross-implementation comparison against BioAge

## Important limitation

This notebook does not resolve EG-004, EG-010, or EG-014. Its outputs remain diagnostic. It does not perform mortality analysis.


In [33]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 190)

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


## 1. Locate the project and load diagnostic data

In [34]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'. "
        "Run this notebook from inside the nhanes project."
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}\n"
        "Run 00_setup_agelens.ipynb first."
    )

CONFIG: dict[str, Any] = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
PROCESSED_ROOT = PROJECT_ROOT / CONFIG["paths"]["processed_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
FIGURES_ROOT = PROJECT_ROOT / CONFIG["paths"]["figures"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
EXTERNAL_ROOT = PROJECT_ROOT / "data" / "external"

for path in [
    PROCESSED_ROOT,
    TABLES_ROOT,
    FIGURES_ROOT,
    LOGS_ROOT,
    EXTERNAL_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed diagnostic input not found: {INPUT_PATH}\n"
        "Run 02_data_preprocessing.ipynb first."
    )

data = pd.read_parquet(INPUT_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_PATH}")
print(f"Rows: {len(data):,}")
print(f"Columns: {len(data.columns)}")


Project root: <PROJECT_ROOT>
Input: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_preprocessed_diagnostic.parquet
Rows: 19,225
Columns: 94


## 2. Validate schema and governance guards

In [35]:
cycle_column = CONFIG["nhanes"]["cycle_column"]
weight_column = CONFIG["nhanes"]["survey_design"]["pooled_weight_variable"]
stratum_column = CONFIG["nhanes"]["survey_design"]["stratum_variable"]
psu_column = CONFIG["nhanes"]["survey_design"]["psu_variable"]

ERRATUM_COLUMN = "harmonized_phenoage_erratum_years"
SUPPLEMENT_COLUMN = "harmonized_phenoage_supplement_years"
AGE_COLUMN = "chronological_age_years"

required_columns = {
    "SEQN",
    cycle_column,
    AGE_COLUMN,
    "age_topcoded",
    "age_below_20",
    "in_fasting_subsample",
    "complete_case_harmonized",
    "complete_case_harmonized_no_topcode",
    "complete_case_harmonized_age20plus",
    "complete_case_bridge_comparison",
    weight_column,
    stratum_column,
    psu_column,
    ERRATUM_COLUMN,
    SUPPLEMENT_COLUMN,
    "prebridge_phenoage_erratum_years",
    "prebridge_phenoage_supplement_years",
    "diagnostic_only",
    "final_scientific_result",
}

missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise ValueError(f"Validation input is missing columns: {missing_columns}")

if data.duplicated([cycle_column, "SEQN"]).any():
    raise ValueError("Duplicate cycle + SEQN combinations found.")

if not data["diagnostic_only"].all():
    raise ValueError("Input contains rows not labeled diagnostic_only.")

if data["final_scientific_result"].any():
    raise ValueError("Input contains rows incorrectly labeled as final results.")

open_gaps = set(CONFIG["governance"]["open_core_evidence_gaps"])
required_open_gaps = {"EG-004", "EG-010", "EG-014"}

if not required_open_gaps.issubset(open_gaps):
    raise ValueError(
        "Expected Core Evidence Gaps are not explicitly recorded."
    )

print("✅ Validation schema and governance guards passed.")


✅ Validation schema and governance guards passed.


## 3. Define validation samples

In [36]:
sample_masks = {
    "all_harmonized_complete_case": data["complete_case_harmonized"],
    "no_topcode": (
        data["complete_case_harmonized"]
        & ~data["age_topcoded"]
    ),
    "age20plus": (
        data["complete_case_harmonized"]
        & data[AGE_COLUMN].ge(20)
    ),
    "age20plus_no_topcode": (
        data["complete_case_harmonized"]
        & data[AGE_COLUMN].ge(20)
        & ~data["age_topcoded"]
    ),
    "bridge_comparison_identical_sample": (
        data["complete_case_bridge_comparison"]
    ),
}

sample_count_records = []

for sample_name, mask in sample_masks.items():
    sample_frame = data.loc[mask]

    for cycle, cycle_frame in sample_frame.groupby(
        cycle_column,
        observed=True,
    ):
        sample_count_records.append(
            {
                "sample": sample_name,
                "cycle": cycle,
                "n": int(len(cycle_frame)),
                "weighted_population_sum": float(
                    cycle_frame[weight_column].sum()
                ),
                "age_topcoded_n": int(
                    cycle_frame["age_topcoded"].sum()
                ),
                "age_below_20_n": int(
                    cycle_frame["age_below_20"].sum()
                ),
            }
        )

sample_counts = pd.DataFrame(sample_count_records)
display(sample_counts)


,sample,cycle,n,weighted_population_sum,age_topcoded_n,age_below_20_n
0,all_harmonized_complete_case,2015_2016,2645,1.292913e+08,121,464
1,all_harmonized_complete_case,2017_2018,2578,1.302142e+08,151,392
2,no_topcode,2015_2016,2524,1.251893e+08,0,464
3,no_topcode,2017_2018,2427,1.252206e+08,0,392
4,age20plus,2015_2016,2181,1.128139e+08,121,0
5,age20plus,2017_2018,2186,1.145595e+08,151,0
6,age20plus_no_topcode,2015_2016,2060,1.087119e+08,0,0
7,age20plus_no_topcode,2017_2018,2035,1.095660e+08,0,0
8,bridge_comparison_identical_sample,2015_2016,2645,1.292913e+08,121,464
9,bridge_comparison_identical_sample,2017_2018,2578,1.302142e+08,151,392


## 4. Weighted statistical helper functions

In [37]:
def _valid_weighted_frame(
    frame: pd.DataFrame,
    columns: list[str],
    weight: str = weight_column,
) -> pd.DataFrame:
    required = [*columns, weight]
    valid = frame.loc[:, required].dropna().copy()
    valid = valid.loc[valid[weight] > 0]
    return valid


def weighted_mean(
    frame: pd.DataFrame,
    value: str,
    weight: str = weight_column,
) -> float:
    valid = _valid_weighted_frame(frame, [value], weight)

    if valid.empty:
        return float("nan")

    return float(
        np.average(valid[value], weights=valid[weight])
    )


def weighted_variance(
    frame: pd.DataFrame,
    value: str,
    weight: str = weight_column,
) -> float:
    valid = _valid_weighted_frame(frame, [value], weight)

    if valid.empty:
        return float("nan")

    mean = np.average(valid[value], weights=valid[weight])
    variance = np.average(
        (valid[value] - mean) ** 2,
        weights=valid[weight],
    )
    return float(variance)


def weighted_sd(
    frame: pd.DataFrame,
    value: str,
    weight: str = weight_column,
) -> float:
    variance = weighted_variance(frame, value, weight)
    return float(np.sqrt(variance))


def weighted_quantile(
    frame: pd.DataFrame,
    value: str,
    quantile: float,
    weight: str = weight_column,
) -> float:
    if not 0 <= quantile <= 1:
        raise ValueError("quantile must be between 0 and 1.")

    valid = _valid_weighted_frame(frame, [value], weight)
    if valid.empty:
        return float("nan")

    valid = valid.sort_values(value)
    cumulative = valid[weight].cumsum()
    cutoff = quantile * valid[weight].sum()

    return float(
        valid.loc[cumulative.ge(cutoff), value].iloc[0]
    )


def weighted_corr(
    frame: pd.DataFrame,
    x: str,
    y: str,
    weight: str = weight_column,
) -> float:
    valid = _valid_weighted_frame(frame, [x, y], weight)

    if len(valid) < 2:
        return float("nan")

    w = valid[weight].to_numpy(dtype=float)
    x_values = valid[x].to_numpy(dtype=float)
    y_values = valid[y].to_numpy(dtype=float)

    x_mean = np.average(x_values, weights=w)
    y_mean = np.average(y_values, weights=w)

    covariance = np.average(
        (x_values - x_mean) * (y_values - y_mean),
        weights=w,
    )
    x_variance = np.average(
        (x_values - x_mean) ** 2,
        weights=w,
    )
    y_variance = np.average(
        (y_values - y_mean) ** 2,
        weights=w,
    )

    if x_variance <= 0 or y_variance <= 0:
        return float("nan")

    return float(
        covariance / np.sqrt(x_variance * y_variance)
    )


def kish_effective_n(
    frame: pd.DataFrame,
    weight: str = weight_column,
) -> float:
    valid = frame.loc[frame[weight].notna() & frame[weight].gt(0)]
    if valid.empty:
        return float("nan")

    weights = valid[weight].to_numpy(dtype=float)
    return float((weights.sum() ** 2) / np.square(weights).sum())


def survey_mean_taylor(
    frame: pd.DataFrame,
    value: str,
    *,
    weight: str = weight_column,
    stratum: str = stratum_column,
    psu: str = psu_column,
) -> dict[str, float]:
    """Taylor-linearized mean and SE for a stratified clustered design.

    The implementation uses with-replacement PSU variance estimation.
    Strata with fewer than two observed PSUs are rejected rather than
    silently approximated.
    """
    valid = frame.loc[
        :,
        [value, weight, stratum, psu],
    ].dropna().copy()

    valid = valid.loc[valid[weight] > 0]

    if valid.empty:
        return {
            "mean": float("nan"),
            "se": float("nan"),
            "ci_low": float("nan"),
            "ci_high": float("nan"),
            "n": 0,
            "psu_count": 0,
            "stratum_count": 0,
        }

    estimate = float(
        np.average(valid[value], weights=valid[weight])
    )
    total_weight = float(valid[weight].sum())

    valid["_linearized"] = (
        valid[weight] * (valid[value] - estimate) / total_weight
    )

    psu_totals = (
        valid.groupby([stratum, psu], observed=True)["_linearized"]
        .sum()
        .reset_index()
    )

    variance = 0.0

    for _, stratum_frame in psu_totals.groupby(
        stratum,
        observed=True,
    ):
        m = len(stratum_frame)

        if m < 2:
            raise RuntimeError(
                "A survey stratum contains fewer than two observed PSUs. "
                "Do not apply an unapproved lonely-PSU correction."
            )

        totals = stratum_frame["_linearized"].to_numpy(dtype=float)
        centered = totals - totals.mean()
        variance += (m / (m - 1)) * float(np.square(centered).sum())

    se = math.sqrt(max(variance, 0.0))
    ci_low = estimate - 1.96 * se
    ci_high = estimate + 1.96 * se

    return {
        "mean": estimate,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n": int(len(valid)),
        "psu_count": int(len(psu_totals)),
        "stratum_count": int(valid[stratum].nunique()),
    }


def weighted_linear_residuals(
    frame: pd.DataFrame,
    outcome: str,
    predictor: str,
    weight: str = weight_column,
) -> pd.Series:
    valid = _valid_weighted_frame(
        frame,
        [outcome, predictor],
        weight,
    )

    residuals = pd.Series(
        np.nan,
        index=frame.index,
        dtype="float64",
    )

    if len(valid) < 3:
        return residuals

    x = np.column_stack(
        [
            np.ones(len(valid)),
            valid[predictor].to_numpy(dtype=float),
        ]
    )
    y = valid[outcome].to_numpy(dtype=float)
    sqrt_w = np.sqrt(valid[weight].to_numpy(dtype=float))

    beta, *_ = np.linalg.lstsq(
        x * sqrt_w[:, None],
        y * sqrt_w,
        rcond=None,
    )

    residuals.loc[valid.index] = y - x @ beta
    return residuals


## 5. Age correlation validation

In [38]:
age_correlation_records = []

for sample_name in [
    "all_harmonized_complete_case",
    "no_topcode",
    "age20plus",
    "age20plus_no_topcode",
]:
    sample_frame = data.loc[sample_masks[sample_name]]

    for cycle, cycle_frame in sample_frame.groupby(
        cycle_column,
        observed=True,
    ):
        for variant_name, phenoage_column in {
            "erratum": ERRATUM_COLUMN,
            "supplement": SUPPLEMENT_COLUMN,
        }.items():
            unweighted_pearson = (
                cycle_frame[[AGE_COLUMN, phenoage_column]]
                .corr(method="pearson")
                .iloc[0, 1]
            )
            unweighted_spearman = (
                cycle_frame[[AGE_COLUMN, phenoage_column]]
                .corr(method="spearman")
                .iloc[0, 1]
            )

            age_correlation_records.append(
                {
                    "sample": sample_name,
                    "cycle": cycle,
                    "formula_variant": variant_name,
                    "n": int(
                        cycle_frame[
                            [AGE_COLUMN, phenoage_column]
                        ]
                        .dropna()
                        .shape[0]
                    ),
                    "pearson_unweighted": float(
                        unweighted_pearson
                    ),
                    "spearman_unweighted": float(
                        unweighted_spearman
                    ),
                    "pearson_weighted_point_estimate": weighted_corr(
                        cycle_frame,
                        AGE_COLUMN,
                        phenoage_column,
                    ),
                    "kish_effective_n": kish_effective_n(
                        cycle_frame
                    ),
                }
            )

age_correlations = pd.DataFrame(age_correlation_records)
display(age_correlations.round(5))


,sample,cycle,formula_variant,n,pearson_unweighted,spearman_unweighted,pearson_weighted_point_estimate,kish_effective_n
0,all_harmonized_complete_case,2015_2016,erratum,2645,0.93935,0.95382,0.94335,1423.37318
1,all_harmonized_complete_case,2015_2016,supplement,2645,0.93935,0.95382,0.94335,1423.37318
2,all_harmonized_complete_case,2017_2018,erratum,2578,0.93875,0.95503,0.94513,1160.65374
3,all_harmonized_complete_case,2017_2018,supplement,2578,0.93875,0.95503,0.94513,1160.65374
4,no_topcode,2015_2016,erratum,2524,0.93425,0.94960,0.93896,1354.36969
5,no_topcode,2015_2016,supplement,2524,0.93425,0.94960,0.93896,1354.36969
6,no_topcode,2017_2018,erratum,2427,0.93073,0.95005,0.93899,1090.61878
7,no_topcode,2017_2018,supplement,2427,0.93073,0.95005,0.93899,1090.61878
8,age20plus,2015_2016,erratum,2181,0.90928,0.93189,0.92556,1177.82925
9,age20plus,2015_2016,supplement,2181,0.90928,0.93189,0.92556,1177.82925


## 6. Survey-weighted descriptive summaries

In [39]:
weighted_summary_records = []

for sample_name in [
    "all_harmonized_complete_case",
    "no_topcode",
    "age20plus",
    "age20plus_no_topcode",
]:
    sample_frame = data.loc[sample_masks[sample_name]]

    for cycle, cycle_frame in sample_frame.groupby(
        cycle_column,
        observed=True,
    ):
        for variant_name, phenoage_column in {
            "erratum": ERRATUM_COLUMN,
            "supplement": SUPPLEMENT_COLUMN,
        }.items():
            survey_result = survey_mean_taylor(
                cycle_frame,
                phenoage_column,
            )

            weighted_summary_records.append(
                {
                    "sample": sample_name,
                    "cycle": cycle,
                    "formula_variant": variant_name,
                    "n": survey_result["n"],
                    "weighted_mean": survey_result["mean"],
                    "taylor_se": survey_result["se"],
                    "ci_low_95": survey_result["ci_low"],
                    "ci_high_95": survey_result["ci_high"],
                    "weighted_sd_descriptive": weighted_sd(
                        cycle_frame,
                        phenoage_column,
                    ),
                    "weighted_median": weighted_quantile(
                        cycle_frame,
                        phenoage_column,
                        0.50,
                    ),
                    "weighted_q25": weighted_quantile(
                        cycle_frame,
                        phenoage_column,
                        0.25,
                    ),
                    "weighted_q75": weighted_quantile(
                        cycle_frame,
                        phenoage_column,
                        0.75,
                    ),
                    "kish_effective_n": kish_effective_n(
                        cycle_frame
                    ),
                    "stratum_count": survey_result[
                        "stratum_count"
                    ],
                    "psu_count": survey_result["psu_count"],
                }
            )

weighted_summaries = pd.DataFrame(weighted_summary_records)
display(weighted_summaries.round(4))


,sample,cycle,formula_variant,n,weighted_mean,taylor_se,ci_low_95,ci_high_95,weighted_sd_descriptive,weighted_median,weighted_q25,weighted_q75,kish_effective_n,stratum_count,psu_count
0,all_harmonized_complete_case,2015_2016,erratum,2645,43.7486,0.8169,42.1474,45.3497,21.0925,43.6792,25.9149,59.9603,1423.3732,15,30
1,all_harmonized_complete_case,2015_2016,supplement,2645,42.1409,0.8304,40.5133,43.7684,21.4398,42.0703,24.0135,58.6197,1423.3732,15,30
2,all_harmonized_complete_case,2017_2018,erratum,2578,44.8160,0.6499,43.5421,46.0899,21.5055,44.2572,26.8197,60.2427,1160.6537,15,30
3,all_harmonized_complete_case,2017_2018,supplement,2578,43.2259,0.6606,41.9310,44.5208,21.8597,42.6579,24.9331,58.9067,1160.6537,15,30
4,no_topcode,2015_2016,erratum,2524,42.4356,0.7677,40.9309,43.9403,20.0593,42.6266,25.4255,58.2738,1354.3697,15,30
5,no_topcode,2015_2016,supplement,2524,40.8063,0.7804,39.2768,42.3358,20.3897,41.0004,23.5160,56.9053,1354.3697,15,30
6,no_topcode,2017_2018,erratum,2427,43.2319,0.5748,42.1053,44.3586,20.3285,42.7405,26.2225,58.4177,1090.6188,15,30
7,no_topcode,2017_2018,supplement,2427,41.6157,0.5843,40.4705,42.7609,20.6633,41.1162,24.3261,57.0516,1090.6188,15,30
8,age20plus,2015_2016,erratum,2181,47.9965,0.8219,46.3856,49.6074,19.0810,47.5572,32.2793,62.2059,1177.8293,15,30
9,age20plus,2015_2016,supplement,2181,46.4587,0.8354,44.8213,48.0962,19.3953,46.0122,30.4827,60.9022,1177.8293,15,30


## 7. Survey-weighted residual PhenoAge acceleration

In [40]:
validation_data = data.copy()

for variant_name, phenoage_column in {
    "erratum": ERRATUM_COLUMN,
    "supplement": SUPPLEMENT_COLUMN,
}.items():
    output_column = (
        f"phenoage_accel_{variant_name}_"
        "weighted_residual_cycle_specific"
    )
    validation_data[output_column] = np.nan

    for cycle, cycle_frame in validation_data.loc[
        sample_masks["all_harmonized_complete_case"]
    ].groupby(cycle_column, observed=True):
        residuals = weighted_linear_residuals(
            cycle_frame,
            phenoage_column,
            AGE_COLUMN,
        )
        validation_data.loc[
            residuals.index,
            output_column,
        ] = residuals

accel_summary_records = []

for cycle, cycle_frame in validation_data.loc[
    sample_masks["all_harmonized_complete_case"]
].groupby(cycle_column, observed=True):
    for variant_name in ["erratum", "supplement"]:
        accel_column = (
            f"phenoage_accel_{variant_name}_"
            "weighted_residual_cycle_specific"
        )

        accel_summary_records.append(
            {
                "cycle": cycle,
                "formula_variant": variant_name,
                "n": int(cycle_frame[accel_column].notna().sum()),
                "weighted_mean_should_be_near_zero": weighted_mean(
                    cycle_frame,
                    accel_column,
                ),
                "weighted_sd": weighted_sd(
                    cycle_frame,
                    accel_column,
                ),
                "age_correlation_weighted_should_be_near_zero": (
                    weighted_corr(
                        cycle_frame,
                        AGE_COLUMN,
                        accel_column,
                    )
                ),
            }
        )

accel_summary = pd.DataFrame(accel_summary_records)
display(accel_summary.round(8))


,cycle,formula_variant,n,weighted_mean_should_be_near_zero,weighted_sd,age_correlation_weighted_should_be_near_zero
0,2015_2016,erratum,2645,-0.0,6.998233,-0.0
1,2015_2016,supplement,2645,0.0,7.113492,-0.0
2,2017_2018,erratum,2578,-0.0,7.026017,-0.0
3,2017_2018,supplement,2578,-0.0,7.141734,-0.0


## 8. Erratum-versus-Supplement agreement

In [41]:
constant_agreement_records = []

for sample_name in [
    "all_harmonized_complete_case",
    "no_topcode",
    "age20plus",
    "age20plus_no_topcode",
]:
    sample_frame = data.loc[sample_masks[sample_name]].copy()

    for cycle, cycle_frame in sample_frame.groupby(
        cycle_column,
        observed=True,
    ):
        difference = (
            cycle_frame[SUPPLEMENT_COLUMN]
            - cycle_frame[ERRATUM_COLUMN]
        )
        average = (
            cycle_frame[SUPPLEMENT_COLUMN]
            + cycle_frame[ERRATUM_COLUMN]
        ) / 2

        mean_difference = weighted_mean(
            cycle_frame.assign(_difference=difference),
            "_difference",
        )
        difference_sd = weighted_sd(
            cycle_frame.assign(_difference=difference),
            "_difference",
        )

        constant_agreement_records.append(
            {
                "sample": sample_name,
                "cycle": cycle,
                "n": int(difference.notna().sum()),
                "mae_unweighted": float(
                    difference.abs().mean()
                ),
                "rmse_unweighted": float(
                    np.sqrt(np.mean(np.square(difference)))
                ),
                "mean_supplement_minus_erratum_weighted": (
                    mean_difference
                ),
                "sd_difference_weighted": difference_sd,
                "bland_altman_lower_weighted_descriptive": (
                    mean_difference - 1.96 * difference_sd
                ),
                "bland_altman_upper_weighted_descriptive": (
                    mean_difference + 1.96 * difference_sd
                ),
                "difference_average_correlation_unweighted": float(
                    np.corrcoef(
                        difference.to_numpy(dtype=float),
                        average.to_numpy(dtype=float),
                    )[0, 1]
                ),
            }
        )

constant_agreement = pd.DataFrame(
    constant_agreement_records
)
display(constant_agreement.round(5))


,sample,cycle,n,mae_unweighted,rmse_unweighted,mean_supplement_minus_erratum_weighted,sd_difference_weighted,bland_altman_lower_weighted_descriptive,bland_altman_upper_weighted_descriptive,difference_average_correlation_unweighted
0,all_harmonized_complete_case,2015_2016,2645,1.58831,1.63326,-1.60770,0.34739,-2.28858,-0.92682,1.0
1,all_harmonized_complete_case,2017_2018,2578,1.55097,1.59789,-1.59012,0.35419,-2.28433,-0.89590,1.0
2,no_topcode,2015_2016,2524,1.61921,1.65877,-1.62932,0.33037,-2.27685,-0.98179,1.0
3,no_topcode,2017_2018,2427,1.58944,1.62999,-1.61621,0.33481,-2.27243,-0.95999,1.0
4,age20plus,2015_2016,2181,1.48275,1.51959,-1.53773,0.31426,-2.15369,-0.92178,1.0
5,age20plus,2017_2018,2186,1.45550,1.49378,-1.52224,0.32140,-2.15217,-0.89230,1.0
6,age20plus_no_topcode,2015_2016,2060,1.51441,1.54628,-1.56000,0.29664,-2.14141,-0.97858,1.0
7,age20plus_no_topcode,2017_2018,2035,1.49430,1.52679,-1.54896,0.30153,-2.13997,-0.95796,1.0


## 9. Pre/post bridging validation

In [42]:
bridge_sample = data.loc[
    sample_masks["bridge_comparison_identical_sample"]
].copy()

bridge_validation_records = []

for cycle, cycle_frame in bridge_sample.groupby(
    cycle_column,
    observed=True,
):
    for variant_name in ["erratum", "supplement"]:
        pre_column = (
            f"prebridge_phenoage_{variant_name}_years"
        )
        post_column = (
            f"harmonized_phenoage_{variant_name}_years"
        )
        difference_column = "_post_minus_pre"

        temporary = cycle_frame.copy()
        temporary[difference_column] = (
            temporary[post_column]
            - temporary[pre_column]
        )

        pre_survey = survey_mean_taylor(
            temporary,
            pre_column,
        )
        post_survey = survey_mean_taylor(
            temporary,
            post_column,
        )
        difference_survey = survey_mean_taylor(
            temporary,
            difference_column,
        )

        bridge_validation_records.append(
            {
                "cycle": cycle,
                "formula_variant": variant_name,
                "n_identical_sample": int(len(temporary)),
                "pre_weighted_mean": pre_survey["mean"],
                "post_weighted_mean": post_survey["mean"],
                "mean_post_minus_pre_weighted": (
                    difference_survey["mean"]
                ),
                "difference_taylor_se": (
                    difference_survey["se"]
                ),
                "difference_ci_low_95": (
                    difference_survey["ci_low"]
                ),
                "difference_ci_high_95": (
                    difference_survey["ci_high"]
                ),
                "weighted_sd_post_minus_pre": weighted_sd(
                    temporary,
                    difference_column,
                ),
            }
        )

bridge_validation = pd.DataFrame(
    bridge_validation_records
)
display(bridge_validation.round(5))


,cycle,formula_variant,n_identical_sample,pre_weighted_mean,post_weighted_mean,mean_post_minus_pre_weighted,difference_taylor_se,difference_ci_low_95,difference_ci_high_95,weighted_sd_post_minus_pre
0,2015_2016,erratum,2645,42.40528,43.74857,1.34329,0.01440,1.31507,1.37150,0.53828
1,2015_2016,supplement,2645,40.77546,42.14087,1.36541,0.01463,1.33673,1.39409,0.54714
2,2017_2018,erratum,2578,44.81603,44.81603,0.00000,0.00000,0.00000,0.00000,0.00000
3,2017_2018,supplement,2578,43.22592,43.22592,0.00000,0.00000,0.00000,0.00000,0.00000


## 10. Cycle comparison after harmonization

In [43]:
cycle_comparison_records = []

for sample_name in [
    "all_harmonized_complete_case",
    "no_topcode",
    "age20plus",
    "age20plus_no_topcode",
]:
    sample_frame = data.loc[sample_masks[sample_name]]

    for variant_name, phenoage_column in {
        "erratum": ERRATUM_COLUMN,
        "supplement": SUPPLEMENT_COLUMN,
    }.items():
        cycle_results = {}

        for cycle, cycle_frame in sample_frame.groupby(
            cycle_column,
            observed=True,
        ):
            cycle_results[cycle] = survey_mean_taylor(
                cycle_frame,
                phenoage_column,
            )

        if set(cycle_results) != {"2015_2016", "2017_2018"}:
            raise RuntimeError(
                f"Both cycles are required for comparison: {sample_name}"
            )

        earlier = cycle_results["2015_2016"]
        later = cycle_results["2017_2018"]

        difference = later["mean"] - earlier["mean"]
        difference_se = math.sqrt(
            earlier["se"] ** 2 + later["se"] ** 2
        )

        cycle_comparison_records.append(
            {
                "sample": sample_name,
                "formula_variant": variant_name,
                "mean_2015_2016": earlier["mean"],
                "se_2015_2016": earlier["se"],
                "mean_2017_2018": later["mean"],
                "se_2017_2018": later["se"],
                "difference_2017_2018_minus_2015_2016": (
                    difference
                ),
                "difference_se_independent_cycles": (
                    difference_se
                ),
                "difference_ci_low_95": (
                    difference - 1.96 * difference_se
                ),
                "difference_ci_high_95": (
                    difference + 1.96 * difference_se
                ),
            }
        )

cycle_comparison = pd.DataFrame(
    cycle_comparison_records
)
display(cycle_comparison.round(5))


,sample,formula_variant,mean_2015_2016,se_2015_2016,mean_2017_2018,se_2017_2018,difference_2017_2018_minus_2015_2016,difference_se_independent_cycles,difference_ci_low_95,difference_ci_high_95
0,all_harmonized_complete_case,erratum,43.74857,0.81692,44.81603,0.64995,1.06747,1.04393,-0.97864,3.11357
1,all_harmonized_complete_case,supplement,42.14087,0.83038,43.22592,0.66065,1.08505,1.06112,-0.99475,3.16485
2,no_topcode,erratum,42.43560,0.76771,43.23194,0.57482,0.79634,0.95906,-1.08341,2.67609
3,no_topcode,supplement,40.80628,0.78035,41.61573,0.58429,0.80945,0.97485,-1.10126,2.72017
4,age20plus,erratum,47.99648,0.82190,48.93740,0.67195,0.94091,1.06161,-1.13985,3.02168
5,age20plus,supplement,46.45875,0.83543,47.41516,0.68301,0.95641,1.07910,-1.15862,3.07144
6,age20plus_no_topcode,erratum,46.64480,0.77060,47.31480,0.62734,0.67001,0.99367,-1.27759,2.61760
7,age20plus_no_topcode,supplement,45.08480,0.78330,45.76584,0.63767,0.68104,1.01004,-1.29863,2.66071


## 11. Export BioAge comparison input

In [44]:
bioage_input_columns = {
    "SEQN": "SEQN",
    cycle_column: cycle_column,
    AGE_COLUMN: "age",
    "albumin_harmonized_g_L": "albumin_g_L",
    "creatinine_harmonized_umol_L": "creatinine_umol_L",
    "glucose_mmol_L": "glucose_mmol_L",
    "crp_harmonized_mg_dL": "crp_mg_dL",
    "lymphocyte_percent": "lymphocyte_percent",
    "mcv_fL": "mcv_fL",
    "rdw_percent": "rdw_percent",
    "alp_harmonized_U_L": "alp_U_L",
    "wbc_1000cells_uL": "wbc_1000cells_uL",
    ERRATUM_COLUMN: "agelens_erratum",
    SUPPLEMENT_COLUMN: "agelens_supplement",
}

bioage_input = (
    data.loc[
        sample_masks["all_harmonized_complete_case"],
        list(bioage_input_columns),
    ]
    .rename(columns=bioage_input_columns)
    .copy()
)

bioage_input_path = TABLES_ROOT / "03_bioage_input.csv"
bioage_input.to_csv(bioage_input_path, index=False)

benchmark_template_path = (
    EXTERNAL_ROOT / "bioage_phenoage_benchmark.csv"
)

if not benchmark_template_path.exists():
    benchmark_template = bioage_input[
        ["SEQN", cycle_column]
    ].copy()
    benchmark_template["bioage_phenoage"] = np.nan
    benchmark_template.to_csv(
        benchmark_template_path,
        index=False,
    )
    print(
        "Created BioAge benchmark template: "
        f"{benchmark_template_path.relative_to(PROJECT_ROOT)}"
    )
else:
    print(
        "Existing BioAge benchmark file preserved: "
        f"{benchmark_template_path.relative_to(PROJECT_ROOT)}"
    )

print(
    "BioAge input exported: "
    f"{bioage_input_path.relative_to(PROJECT_ROOT)}"
)


Existing BioAge benchmark file preserved: data\external\bioage_phenoage_benchmark.csv
BioAge input exported: results\tables\03_bioage_input.csv


## 12. Optional BioAge cross-implementation comparison

Populate:

```text
nhanes/data/external/bioage_phenoage_benchmark.csv
```

with these columns:

```text
SEQN
NHANES_CYCLE
bioage_phenoage
```

The notebook compares BioAge only when at least one non-missing benchmark value is present.


In [45]:
bioage_comparison = pd.DataFrame()
bioage_bland_altman = pd.DataFrame()

if benchmark_template_path.exists():
    benchmark = pd.read_csv(benchmark_template_path)

    required_benchmark_columns = {
        "SEQN",
        cycle_column,
        "bioage_phenoage",
    }
    missing_benchmark_columns = (
        required_benchmark_columns - set(benchmark.columns)
    )

    if missing_benchmark_columns:
        raise ValueError(
            "BioAge benchmark file is missing columns: "
            f"{sorted(missing_benchmark_columns)}"
        )

    benchmark["SEQN"] = pd.to_numeric(
        benchmark["SEQN"],
        errors="raise",
    ).astype("Int64")

    benchmark["bioage_phenoage"] = pd.to_numeric(
        benchmark["bioage_phenoage"],
        errors="coerce",
    )

    if benchmark.duplicated([cycle_column, "SEQN"]).any():
        raise ValueError(
            "BioAge benchmark contains duplicate cycle + SEQN rows."
        )

    benchmark_nonmissing = benchmark.loc[
        benchmark["bioage_phenoage"].notna()
    ].copy()

    if benchmark_nonmissing.empty:
        print(
            "BioAge benchmark template exists but contains no results yet. "
            "Cross-implementation comparison was skipped."
        )
    else:
        comparison = bioage_input.merge(
            benchmark_nonmissing,
            on=["SEQN", cycle_column],
            how="inner",
            validate="one_to_one",
        )

        comparison_records = []
        bland_altman_records = []

        for cycle, cycle_frame in comparison.groupby(
            cycle_column,
            observed=True,
        ):
            for variant_name, agelens_column in {
                "erratum": "agelens_erratum",
                "supplement": "agelens_supplement",
            }.items():
                difference = (
                    cycle_frame[agelens_column]
                    - cycle_frame["bioage_phenoage"]
                )
                average = (
                    cycle_frame[agelens_column]
                    + cycle_frame["bioage_phenoage"]
                ) / 2

                mean_difference = float(difference.mean())
                difference_sd = float(difference.std(ddof=1))

                comparison_records.append(
                    {
                        "cycle": cycle,
                        "agelens_variant": variant_name,
                        "n_identical_sample": int(len(cycle_frame)),
                        "mae": float(difference.abs().mean()),
                        "rmse": float(
                            np.sqrt(np.mean(np.square(difference)))
                        ),
                        "mean_agelens_minus_bioage": (
                            mean_difference
                        ),
                        "pearson": float(
                            cycle_frame[
                                [agelens_column, "bioage_phenoage"]
                            ]
                            .corr(method="pearson")
                            .iloc[0, 1]
                        ),
                        "spearman": float(
                            cycle_frame[
                                [agelens_column, "bioage_phenoage"]
                            ]
                            .corr(method="spearman")
                            .iloc[0, 1]
                        ),
                    }
                )

                bland_altman_records.append(
                    {
                        "cycle": cycle,
                        "agelens_variant": variant_name,
                        "mean_difference": mean_difference,
                        "lower_limit": (
                            mean_difference - 1.96 * difference_sd
                        ),
                        "upper_limit": (
                            mean_difference + 1.96 * difference_sd
                        ),
                        "difference_average_correlation": float(
                            np.corrcoef(
                                difference.to_numpy(dtype=float),
                                average.to_numpy(dtype=float),
                            )[0, 1]
                        ),
                    }
                )

        bioage_comparison = pd.DataFrame(comparison_records)
        bioage_bland_altman = pd.DataFrame(
            bland_altman_records
        )

        display(bioage_comparison.round(6))
        display(bioage_bland_altman.round(6))


,cycle,agelens_variant,n_identical_sample,mae,rmse,mean_agelens_minus_bioage,pearson,spearman
0,2015_2016,erratum,2645,1.638000,1.679885,1.637691,1.0,1.0
1,2015_2016,supplement,2645,0.049726,0.051009,0.049726,1.0,1.0
2,2017_2018,erratum,2578,1.601316,1.644821,1.600600,1.0,1.0
3,2017_2018,supplement,2578,0.050348,0.051643,0.050337,1.0,1.0


,cycle,agelens_variant,mean_difference,lower_limit,upper_limit,difference_average_correlation
0,2015_2016,erratum,1.637691,0.904236,2.371146,-0.999766
1,2015_2016,supplement,0.049726,0.027437,0.072014,0.696215
2,2017_2018,erratum,1.600600,0.857934,2.343265,-0.999788
3,2017_2018,supplement,0.050337,0.027706,0.072967,0.731621


## 13. Diagnostic figures

In [46]:
figure_data = data.loc[
    sample_masks["all_harmonized_complete_case"]
].copy()

# Age versus Phenotypic Age by cycle.
for cycle, cycle_frame in figure_data.groupby(
    cycle_column,
    observed=True,
):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        cycle_frame[AGE_COLUMN],
        cycle_frame[ERRATUM_COLUMN],
        alpha=0.35,
        s=12,
    )

    lower = float(
        min(
            cycle_frame[AGE_COLUMN].min(),
            cycle_frame[ERRATUM_COLUMN].min(),
        )
    )
    upper = float(
        max(
            cycle_frame[AGE_COLUMN].max(),
            cycle_frame[ERRATUM_COLUMN].max(),
        )
    )

    ax.plot([lower, upper], [lower, upper], linestyle="--")
    ax.set_xlabel("Chronological age (years)")
    ax.set_ylabel("Diagnostic Phenotypic Age — erratum (years)")
    ax.set_title(f"AgeLens diagnostic age comparison — {cycle}")
    fig.tight_layout()

    figure_path = (
        FIGURES_ROOT
        / f"03_age_vs_phenoage_erratum_{cycle}.png"
    )
    fig.savefig(figure_path, dpi=200)
    plt.close(fig)

# Erratum-versus-Supplement Bland–Altman figure.
for cycle, cycle_frame in figure_data.groupby(
    cycle_column,
    observed=True,
):
    average = (
        cycle_frame[ERRATUM_COLUMN]
        + cycle_frame[SUPPLEMENT_COLUMN]
    ) / 2
    difference = (
        cycle_frame[SUPPLEMENT_COLUMN]
        - cycle_frame[ERRATUM_COLUMN]
    )

    mean_difference = float(difference.mean())
    sd_difference = float(difference.std(ddof=1))

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        average,
        difference,
        alpha=0.35,
        s=12,
    )
    ax.axhline(mean_difference)
    ax.axhline(
        mean_difference - 1.96 * sd_difference,
        linestyle="--",
    )
    ax.axhline(
        mean_difference + 1.96 * sd_difference,
        linestyle="--",
    )
    ax.set_xlabel("Mean of erratum and Supplement values (years)")
    ax.set_ylabel("Supplement − erratum (years)")
    ax.set_title(f"Formula-constant agreement — {cycle}")
    fig.tight_layout()

    figure_path = (
        FIGURES_ROOT
        / f"03_bland_altman_constants_{cycle}.png"
    )
    fig.savefig(figure_path, dpi=200)
    plt.close(fig)

print("✅ Diagnostic figures saved.")


✅ Diagnostic figures saved.


## 14. Save validation outputs

In [47]:
written_files: list[Path] = []

output_tables = {
    "03_validation_sample_counts.csv": sample_counts,
    "03_age_correlations.csv": age_correlations,
    "03_survey_weighted_summaries.csv": weighted_summaries,
    "03_weighted_acceleration_summary.csv": accel_summary,
    "03_formula_constant_agreement.csv": constant_agreement,
    "03_bridge_validation.csv": bridge_validation,
    "03_cycle_comparison.csv": cycle_comparison,
}

for filename, frame in output_tables.items():
    path = TABLES_ROOT / filename
    frame.to_csv(path, index=False)
    written_files.append(path)

if not bioage_comparison.empty:
    path = TABLES_ROOT / "03_bioage_comparison.csv"
    bioage_comparison.to_csv(path, index=False)
    written_files.append(path)

if not bioage_bland_altman.empty:
    path = TABLES_ROOT / "03_bioage_bland_altman.csv"
    bioage_bland_altman.to_csv(path, index=False)
    written_files.append(path)

validation_output_path = (
    PROCESSED_ROOT
    / "agelens_validation_diagnostic.parquet"
)

validation_columns = [
    "SEQN",
    cycle_column,
    AGE_COLUMN,
    "age_topcoded",
    "age_below_20",
    weight_column,
    stratum_column,
    psu_column,
    ERRATUM_COLUMN,
    SUPPLEMENT_COLUMN,
    "phenoage_accel_erratum_weighted_residual_cycle_specific",
    "phenoage_accel_supplement_weighted_residual_cycle_specific",
    "diagnostic_only",
    "final_scientific_result",
]

validation_data.loc[
    sample_masks["all_harmonized_complete_case"],
    validation_columns,
].to_parquet(validation_output_path, index=False)

written_files.append(validation_output_path)
written_files.append(bioage_input_path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "03_validation.ipynb",
    "input": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "validation_domains": [
        "sample_flow",
        "age_correlation",
        "survey_weighted_descriptive_statistics",
        "formula_constant_agreement",
        "bridging_effect",
        "topcode_sensitivity",
        "age20plus_sensitivity",
        "optional_bioage_cross_implementation",
    ],
    "survey_weight": weight_column,
    "survey_stratum": stratum_column,
    "survey_psu": psu_column,
    "weighted_correlation_status": (
        "point_estimate_only; no design-based standard error"
    ),
    "survey_mean_variance_method": (
        "Taylor linearization with stratified PSU totals"
    ),
    "lonely_psu_correction_applied": False,
    "mortality_data_used": False,
    "bioage_results_present": bool(
        not bioage_comparison.empty
    ),
    "outputs_diagnostic_only": True,
    "final_scientific_results_allowed": False,
    "open_core_evidence_gaps": sorted(open_gaps),
    "outputs": [
        str(path.relative_to(PROJECT_ROOT))
        for path in written_files
    ],
}

metadata_path = LOGS_ROOT / "03_validation_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
written_files.append(metadata_path)

print("Files written:")
for path in written_files:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")


Files written:
  - results\tables\03_validation_sample_counts.csv
  - results\tables\03_age_correlations.csv
  - results\tables\03_survey_weighted_summaries.csv
  - results\tables\03_weighted_acceleration_summary.csv
  - results\tables\03_formula_constant_agreement.csv
  - results\tables\03_bridge_validation.csv
  - results\tables\03_cycle_comparison.csv
  - results\tables\03_bioage_comparison.csv
  - results\tables\03_bioage_bland_altman.csv
  - data\processed\agelens_validation_diagnostic.parquet
  - results\tables\03_bioage_input.csv
  - logs\03_validation_metadata.json


## 15. Final verification

In [48]:
reloaded_validation = pd.read_parquet(
    PROCESSED_ROOT
    / "agelens_validation_diagnostic.parquet"
)

expected_n = int(
    sample_masks["all_harmonized_complete_case"].sum()
)

assert len(reloaded_validation) == expected_n
assert not reloaded_validation.duplicated(
    [cycle_column, "SEQN"]
).any()
assert reloaded_validation["diagnostic_only"].all()
assert not reloaded_validation[
    "final_scientific_result"
].any()
assert reloaded_validation[
    "phenoage_accel_erratum_weighted_residual_cycle_specific"
].notna().all()

for cycle, cycle_frame in reloaded_validation.groupby(
    cycle_column,
    observed=True,
):
    weighted_accel_mean = weighted_mean(
        cycle_frame,
        "phenoage_accel_erratum_weighted_residual_cycle_specific",
    )
    weighted_accel_age_corr = weighted_corr(
        cycle_frame,
        AGE_COLUMN,
        "phenoage_accel_erratum_weighted_residual_cycle_specific",
    )

    if abs(weighted_accel_mean) > 1e-8:
        raise RuntimeError(
            f"Weighted acceleration mean is not near zero for {cycle}: "
            f"{weighted_accel_mean}"
        )

    if abs(weighted_accel_age_corr) > 1e-8:
        raise RuntimeError(
            f"Weighted acceleration remains correlated with age for {cycle}: "
            f"{weighted_accel_age_corr}"
        )

print("✅ Validation outputs verified.")
print("Mortality data were not used.")
print("BioAge comparison runs only after benchmark values are supplied.")
print("All outputs remain diagnostic only.")


✅ Validation outputs verified.
Mortality data were not used.
BioAge comparison runs only after benchmark values are supplied.
All outputs remain diagnostic only.
